# 02 · Detector (S2, S4, S6)

**Historias:** US-02.2 (entrenar), US-02.3 (evaluar), US-02.8 (recall en aves difíciles), SP-02.3b (spike `s`).
**Kernel:** `colibri-yolo`. **Necesita** haber ejecutado el notebook 01 (datos en `datos/`).

| Sección | Qué reproduce | Decisión | Referencia |
|---|---|---|---|
| 1 | Evaluador común (AP de 101 puntos) y línea base COCO `bird` en la prueba vv | — | yolo26n AP50 0,70, R 0,52 |
| 2 | Entrenamiento S2: `yolo26n` con ds2+ds1 y ablación solo ds2 | **D-05** `yolo26n_ds2_ds1` | vv AP50 0,795, P 0,843, R 0,862 |
| 3 | Ajuste fino S4 con ds2+ds1+humming ×3 | **D-08** `yolo26n_ft_v2` | `test_` mAP50 0,706 → 0,877 |
| 4 | Gate viejo frente a nuevo con GT v1 y v2 | D-08 | `resultados/S4_reentreno_v2/comparacion.csv` |
| 5 | Spike `yolo26s` (S6) | **D-11** no adoptar | `resultados/S6_spike_yolo26s/*/veredicto.json` |

**`ENTRENAR`** (por defecto `False` en modo `artefactos`): con `False` se usan los pesos congelados
(`modelos/yolo26n_ft_v2/best.pt` en el repo y `yolo26n_ds2_ds1/best.pt` en la zona de trabajo). Con `True` se
entrena de nuevo: ~1 h para S2 y ~7 h para S4 en la laptop (RTX de 6 GB). El entrenamiento se lanza con la CLI `yolo`
en un proceso aparte, porque en Windows los *workers* del DataLoader necesitan un módulo `__main__` importable.
Es reanudable: si se corta, vuelve a ejecutar la celda.

In [ ]:
%run 00_comun.ipynb
OUT = RUNS / "02"; OUT.mkdir(parents=True, exist_ok=True)
ENTRENAR = MODO == "recalcular"
DATA_FINAL = P["roboflow"] / "final"
TEST_VIDEO = P["frames"] / "vv_prueba"
LISTAS = {v: P["frames"] / f"listas_{v}" for v in ("v1", "v2")}
import subprocess, datetime as dt
from collections import defaultdict
from PIL import Image
from ultralytics import YOLO
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
COCO_BIRD_ID = 14
# read_labels: celda 10 del notebook original de S1
def read_labels(lbl_path: Path) -> list:
    if not lbl_path.exists():
        return []
    rows = []
    for line in lbl_path.read_text().splitlines():
        parts = line.split()
        if len(parts) == 5:
            c, x, y, w, h = parts
            rows.append((int(c), float(x), float(y), float(w), float(h)))
        elif len(parts) > 5:  # polígono: se convierte a caja envolvente
            c, pts = int(parts[0]), np.array(parts[1:], float).reshape(-1, 2)
            (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
            rows.append((c, (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0))
    return rows

## 1. Evaluador común y línea base (S2, CA-02.2.1)

AP50 de 101 puntos estilo COCO a IoU 0,5, P y R a conf 0,25, y recall por tamaño de caja. La línea base es YOLO26n
preentrenado en COCO, clase `bird`, sin ajustar. **Nota:** este `iou_matrix` (con ε) sustituye al de `00_comun` dentro
de este notebook; dan lo mismo salvo en cajas degeneradas.

In [ ]:
# Portado de notebook original colibri_yolo26_tracking.ipynb, celda 22
def iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """IoU entre cajas xyxy: a (N,4), b (M,4)."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    tl = np.maximum(a[:, None, :2], b[None, :, :2]); br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(br - tl, 0, None).prod(2)
    area_a = (a[:, 2:] - a[:, :2]).prod(1); area_b = (b[:, 2:] - b[:, :2]).prod(1)
    return inter / (area_a[:, None] + area_b[None] - inter + 1e-9)

def size_bin(w_px, h_px):
    s = np.sqrt(w_px * h_px)
    return "pequeña(<32px)" if s < 32 else ("mediana(32-96px)" if s < 96 else "grande(>=96px)")

def evaluate(pred_fn, split_dir: Path, iou_thr=0.5, conf_thr=0.25):
    """pred_fn(path) -> (K,5) [x1,y1,x2,y2,conf] en píxeles. Devuelve métricas y tabla por imagen."""
    scores, tps, n_gt, per_img, gt_rows = [], [], 0, [], []
    for img in sorted(p for p in (split_dir / "images").iterdir() if p.suffix.lower() in IMG_EXTS):
        with Image.open(img) as im: W, H = im.size
        gt = np.array([[(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]
                       for _, x, y, w, h in read_labels(split_dir / "labels" / f"{img.stem}.txt")]).reshape(-1, 4)
        pr = np.asarray(pred_fn(img), float).reshape(-1, 5)
        pr = pr[np.argsort(-pr[:, 4])]
        ious = iou_matrix(pr[:, :4], gt); used = np.zeros(len(gt), bool); tp = np.zeros(len(pr), bool)
        for i in range(len(pr)):
            if len(gt):
                j = int(np.argmax(np.where(used, -1, ious[i])))
                if not used[j] and ious[i, j] >= iou_thr:
                    used[j] = tp[i] = True
        # recall a conf_thr por caja GT (para análisis por tamaño)
        keep = pr[:, 4] >= conf_thr
        ious_k = iou_matrix(pr[keep, :4], gt)
        for j, g in enumerate(gt):
            gt_rows.append(dict(path=str(img), size=size_bin(g[2] - g[0], g[3] - g[1]),
                                hit=bool(ious_k.shape[0] and ious_k[:, j].max() >= iou_thr)))
        scores += pr[:, 4].tolist(); tps += tp.tolist(); n_gt += len(gt)
        per_img.append(dict(path=str(img), n_gt=len(gt), n_pred=int(keep.sum()), tp=int((tp & keep).sum())))
    scores, tps = np.array(scores), np.array(tps, bool)
    order = np.argsort(-scores); tp_c = np.cumsum(tps[order]); fp_c = np.cumsum(~tps[order])
    rec = tp_c / max(n_gt, 1); prec = tp_c / np.maximum(tp_c + fp_c, 1)
    # AP de 101 puntos estilo COCO: precisión máxima con recall >= r; 0 si r no se alcanza
    ap50 = float(np.mean([prec[rec >= r].max() if (rec >= r).any() else 0.0 for r in np.linspace(0, 1, 101)]))
    k = scores >= conf_thr
    P = float(tps[k].sum() / max(k.sum(), 1)); R = float(tps[k].sum() / max(n_gt, 1))
    gt_df, img_df = pd.DataFrame(gt_rows), pd.DataFrame(per_img)
    by_size = gt_df.groupby("size")["hit"].agg(["mean", "size"]).rename(columns={"mean": "recall", "size": "n"})
    return {"AP50": ap50, f"P@{conf_thr}": P, f"R@{conf_thr}": R, "n_gt": n_gt}, by_size, img_df

def yolo_pred_fn(model, classes=None, class_map=None, imgsz=640):
    def fn(path):
        r = model.predict(str(path), conf=0.001, imgsz=imgsz, classes=classes, verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0:
            return np.zeros((0, 5))
        return np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])
    return fn

In [ ]:
baseline = YOLO("yolo26n.pt")
base_metrics, base_by_size, base_imgs = evaluate(yolo_pred_fn(baseline, classes=[COCO_BIRD_ID]), TEST_VIDEO)
print("Línea base COCO 'bird' en la prueba vv:", {k: round(v, 3) for k, v in base_metrics.items()}, "| S2: AP50 0,70, R 0,52")
display(base_by_size)

## 2. Entrenamiento S2 (D-05)

`yolo26n.pt` sobre ds2+ds1 (`final/data.yaml`) y, como ablación, solo ds2 (`final_ds2/data.yaml`); 40 épocas,
paciencia 15, `batch=0.70` (fracción de VRAM), `cache=disk` (en Windows `ram` se duplica en cada *worker* y agota la
memoria), 3 *workers*, semilla 42, determinista.

In [ ]:
def entrenar(nombre, pesos_ini, data, **kw):
    """Entrena o reanuda con la CLI de Ultralytics en otro proceso. Devuelve la ruta de best.pt."""
    run = OUT / nombre; done = run / "TRAINING_DONE"; last = run / "weights" / "last.pt"
    if done.exists():
        return run / "weights" / "best.pt"
    if last.exists():
        args = ["train", "resume", f"model={last}"]
    else:
        args = ["detect", "train", f"model={pesos_ini}", f"data={data}", f"project={OUT}", f"name={nombre}", "exist_ok=False",
                *[f"{k}={v}" for k, v in kw.items()]]
    print("yolo", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", "ultralytics", *args], check=True)
    done.write_text(dt.datetime.now().isoformat(timespec="seconds"))
    return run / "weights" / "best.pt"

TRAIN_S2 = dict(epochs=40, imgsz=640, batch=0.70, cache="disk", amp=True, workers=3, patience=15, seed=SEED, deterministic=True, plots=True)
if ENTRENAR:
    TRAINED = {"yolo26n_ds2_ds1": entrenar("yolo26n_ds2_ds1", "yolo26n.pt", DATA_FINAL / "data.yaml", **TRAIN_S2),
               "yolo26n_ds2": entrenar("yolo26n_ds2", "yolo26n.pt", P["roboflow"] / "final_ds2" / "data.yaml", **TRAIN_S2)}
else:
    TRAINED = {"yolo26n_ds2_ds1": DETECTOR_ANTERIOR}
    abl = P["modelos"] / "yolo26n_ds2" / "best.pt"
    if abl.exists():
        TRAINED["yolo26n_ds2"] = abl
print({k: str(v) for k, v in TRAINED.items()})

In [ ]:
REF_S2 = {"yolo26n_ds2_ds1": {"AP50": 0.795, "P@0.25": 0.843, "R@0.25": 0.862}, "yolo26n_ds2": {"AP50": 0.790}}
rows = {"linea_base_coco_bird": base_metrics}
for name, w in TRAINED.items():
    rows[name], by_size, _ = evaluate(yolo_pred_fn(YOLO(str(w)), imgsz=640), TEST_VIDEO)
    for k, ref in REF_S2.get(name, {}).items():
        d = abs(rows[name][k] - ref)
        print(f"{'✓' if d <= TOL['metrica'] else '✗'} {name} {k}: {rows[name][k]:.3f} (S2 {ref})")
COMP_S2 = pd.DataFrame(rows).T; COMP_S2.to_csv(OUT / "comparacion_s2_vv.csv"); display(COMP_S2.round(3))

## 3. Ajuste fino S4 (D-08)

Desde `yolo26n_ds2_ds1` con ds2+ds1 y humming ×3 (`listas_v1/data_ft.yaml`): 20 épocas, paciencia 8, `lr0` 0,002 con
SGD, `warmup_epochs` 1, semilla 0: los parámetros de `historico/scripts/entrenar_ft.py`, lanzados con `entrenar`.

In [ ]:
TRAIN_FT = dict(epochs=20, patience=8, imgsz=640, warmup_epochs=1, lr0=0.002, optimizer="SGD", cache="disk", workers=3, batch=0.70, seed=0)
FT = entrenar("yolo26n_ft_v2", TRAINED["yolo26n_ds2_ds1"], LISTAS["v1"] / "data_ft.yaml", **TRAIN_FT) if ENTRENAR else DETECTOR
print("detector ajustado:", FT)

## 4. Gate: viejo frente a nuevo en `test_` (GT v1 y GT v2)

`evaluar.py` de S4: métricas de Ultralytics (conf 0,001, IoU 0,6) más P y R a conf 0,25 con emparejamiento voraz a
IoU 0,5, global y por video, en `test_video` y en ds2 test (para comprobar que no se olvida ds2).
Criterio fijado antes de entrenar: subir el recall en `test_` sin perder más de 2 puntos de mAP50 en ds2.

In [ ]:
# Portado de video_eval/reentreno/v2/evaluar.py
def iou(a, b):
    x1, y1 = np.maximum(a[:2], b[:2]); x2, y2 = np.minimum(a[2:], b[2:])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)


def gt_boxes(img, w, h):
    lab = Path(img).parent.parent / "labels" / f"{Path(img).stem}.txt"
    out = []
    for l in (lab.read_text().splitlines() if lab.exists() else []):
        p = l.split()
        if len(p) == 5:
            _, x, y, bw, bh = map(float, p)
            out.append(np.array([(x - bw / 2) * w, (y - bh / 2) * h, (x + bw / 2) * w, (y + bh / 2) * h]))
    return out


def pr_at(model, imgs, conf=0.25, imgsz=640):
    """TP/FP/FN a conf fija, agrupados por video (prefijo antes de _fNNNNN)."""
    st = defaultdict(lambda: [0, 0, 0])
    for i in range(0, len(imgs), 16):
        for r in model.predict(imgs[i:i + 16], imgsz=imgsz, conf=conf, verbose=False):
            h, w = r.orig_shape
            gts = gt_boxes(r.path, w, h)
            preds = sorted(zip(r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy()), key=lambda z: -z[0])
            used, tp = set(), 0
            for _, pb in preds:
                best = max(((iou(pb, g), j) for j, g in enumerate(gts) if j not in used), default=(0, -1))
                if best[0] >= 0.5:
                    used.add(best[1]); tp += 1
            vid = re.sub(r"_f\d{5}$", "", Path(r.path).stem)
            for k in (vid, "GLOBAL"):
                st[k][0] += tp; st[k][1] += len(preds) - tp; st[k][2] += len(gts) - tp
    return st


def per_video_ap50(model, imgs, tmp):
    """mAP50 por video con val() de Ultralytics sobre listas temporales."""
    groups = defaultdict(list)
    for p in imgs:
        groups[re.sub(r"_f\d{5}$", "", Path(p).stem)].append(p)
    res = {}
    for vid, ps in groups.items():
        lst = tmp / f"{vid[:40]}.txt"; lst.write_text("\n".join(ps) + "\n")
        y = tmp / f"{vid[:40]}.yaml"; y.write_text(f"train: {lst}\nval: {lst}\nnames:\n  0: hummingbird\n")
        m = model.val(data=str(y), split="val", imgsz=640, conf=0.001, iou=0.6, batch=8, plots=False, verbose=False,
                      project=str(tmp / "runs"), name="pv", exist_ok=True)
        res[vid] = (m.box.mp, m.box.mr, m.box.map50, m.box.map)
    return res

def evaluar_modelos(weights: dict, version: str):
    """main() de evaluar.py con las listas de la versión de GT."""
    L = LISTAS[version]
    SETS = {"test_video": L / "data_test_video.yaml", "ds2_test": L / "data_ds2_test.yaml"}
    tmp = OUT / "_eval_tmp"; tmp.mkdir(exist_ok=True)
    imgs = {"test_video": (L / "test_video.txt").read_text().split(),
            "ds2_test": sorted(str(p) for p in (P["roboflow"] / "final_ds2" / "test" / "images").glob("*.jpg"))}
    rows = []
    for tag, w in weights.items():
        model = YOLO(str(w))
        for name, y in SETS.items():
            m = model.val(data=str(y), split="test", imgsz=640, conf=0.001, iou=0.6, batch=8, plots=False,
                          verbose=False, project=str(tmp / "runs"), name=f"{tag}_{name}", exist_ok=True)
            st = pr_at(model, imgs[name])
            tp, fp, fn = st["GLOBAL"]
            rows.append(dict(modelo=tag, conjunto=name, video="GLOBAL", P=m.box.mp, R=m.box.mr, mAP50=m.box.map50,
                             mAP50_95=m.box.map, P_c025=tp / max(1, tp + fp), R_c025=tp / max(1, tp + fn),
                             n_img=len(imgs[name]), n_gt=tp + fn))
            if name == "test_video":
                for vid, (p, r, m50, m95) in per_video_ap50(model, imgs[name], tmp).items():
                    tp, fp, fn = st[vid]
                    rows.append(dict(modelo=tag, conjunto=name, video=vid, P=p, R=r, mAP50=m50, mAP50_95=m95,
                                     P_c025=tp / max(1, tp + fp), R_c025=tp / max(1, tp + fn),
                                     n_img=sum(1 for x in imgs[name] if Path(x).stem.startswith(vid)), n_gt=tp + fn))
    return pd.DataFrame(rows)

REF4 = P["resultados"] / "S4_reentreno_v2"
GATE = {}
for version, ref in (("v1", "comparacion.csv"), ("v2", "deteccion_gt2.csv")):
    GATE[version] = evaluar_modelos({"viejo": TRAINED["yolo26n_ds2_ds1"], "nuevo": FT}, version)
    GATE[version].to_csv(OUT / f"gate_{version}.csv", index=False)
    display(GATE[version][GATE[version].video == "GLOBAL"].round(3))
    if (REF4 / ref).exists():
        comparar(GATE[version], REF4 / ref, ["modelo", "conjunto", "video"], ["mAP50", "R_c025", "P_c025"], nombre=f"gate GT {version} frente a S4")

## 5. Spike `yolo26s` (S6, D-11)

Se probó `yolo26s` con las mismas listas (`yolo26s_v2`) y con ajuste fino solo en humming (`yolo26s_ft_v2`).
Ninguno pasó el gate. Los pesos (20 MB cada uno) se quedan fuera del repo; los scripts están en
`historico/scripts/comparar_n_vs_s.py` y `comparar_ft.py`. Aquí se muestran los veredictos congelados.

In [ ]:
for f in sorted((P["resultados"] / "S6_spike_yolo26s").glob("*/veredicto.json")):
    v = json.loads(f.read_text(encoding="utf-8"))
    print(f.parent.name, "| adoptar:", v.get("adoptar_yolo26s", v.get("adoptar")), "|", v.get("comprobaciones"))

## Conclusiones

- **S2 (D-05):** `yolo26n_ds2_ds1` sube de la línea base COCO (AP50 0,70, R 0,52) a AP50 0,795, P 0,843 y R 0,862
  en vv. Solo ds2 da 0,790: ds1 aporta poco.
- **S4 (D-08):** con humming, `yolo26n_ft_v2` sube en `test_` de mAP50 0,706 a 0,877 y de R 0,757 a 0,821, y pierde
  0,5 puntos en ds2. **Es el detector vigente.**
- **S6 (D-11):** `yolo26s` no mejora: el cuello de botella son los datos del dominio, no el tamaño del modelo.
- **Pendiente (US-02.8):** el recall en aves pequeñas o de espaldas (J4PK) sigue siendo la debilidad.